# Module 6: Scheduler 调度逻辑

## 学习目标
- 理解 Scheduler 的核心职责
- 掌握 Prefill 和 Decode 批处理
- 学习 Chunked Prefill 策略
- 理解 Continuous Batching

---

## 6.1 代码位置

```
mini-sglang/python/minisgl/scheduler/
├── scheduler.py   # 主调度器
├── config.py      # 调度配置
├── prefill.py     # Prefill 管理
├── decode.py      # Decode 管理
├── table.py       # Table 管理器
├── cache.py       # Cache 包装器
└── io.py          # ZMQ I/O
```

## 6.2 Scheduler 的核心职责

```
┌─────────────────────────────────────────────────────────────────────────┐
│                          Scheduler 职责                                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  1. 接收请求                                                            │
│     - 从 Tokenizer 接收 UserMsg                                        │
│     - 解析 input_ids 和 sampling_params                                │
│                                                                         │
│  2. 资源管理                                                            │
│     - TableManager: 管理 page table 槽位                                │
│     - CacheManager: 管理 KV Cache 分配                                  │
│                                                                         │
│  3. 批处理调度                                                          │
│     - PrefillManager: 调度 prefill 批次                                 │
│     - DecodeManager: 调度 decode 批次                                   │
│                                                                         │
│  4. 执行推理                                                            │
│     - 调用 Engine.forward_batch()                                       │
│     - 处理采样结果                                                      │
│                                                                         │
│  5. 返回结果                                                            │
│     - 发送 DetokenizeMsg 到 Detokenizer                                 │
│     - 释放已完成请求的资源                                              │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
import torch
from dataclasses import dataclass, field
from typing import List, Set, Optional

# 复用之前定义的类
@dataclass
class SamplingParams:
    top_k: int = 1
    temperature: float = 0.0
    max_tokens: int = 1024

@dataclass
class PendingReq:
    """等待调度的请求"""
    uid: int
    input_ids: torch.Tensor
    sampling_params: SamplingParams
    chunked_req: Optional['Req'] = None  # 如果是分块请求
    
    @property
    def input_len(self) -> int:
        return len(self.input_ids)
    
    @property
    def output_len(self) -> int:
        return self.sampling_params.max_tokens

## 6.3 TableManager: 页表槽位管理

管理每个请求在 page table 中的槽位。

In [ ]:
class TableManager:
    """管理 page table 槽位"""
    
    def __init__(self, max_running_req: int, page_table: torch.Tensor):
        self.max_running_req = max_running_req
        self.page_table = page_table
        self.free_slots = list(range(max_running_req))
        
        # Token pool: 存储每个请求的 token IDs
        max_seq_len = page_table.shape[1]
        self.token_pool = torch.zeros(
            (max_running_req, max_seq_len),
            dtype=torch.int32,
            device=page_table.device
        )
    
    @property
    def available_size(self) -> int:
        return len(self.free_slots)
    
    def allocate(self) -> int:
        """分配一个槽位"""
        if not self.free_slots:
            raise RuntimeError("No free slots")
        return self.free_slots.pop(0)
    
    def free(self, table_idx: int) -> None:
        """释放槽位"""
        self.free_slots.append(table_idx)

# 演示
page_table = torch.zeros((256, 4096), dtype=torch.int32)
table_mgr = TableManager(256, page_table)

print(f"初始可用槽位: {table_mgr.available_size}")

# 分配几个槽位
slots = [table_mgr.allocate() for _ in range(5)]
print(f"分配了槽位: {slots}")
print(f"剩余可用槽位: {table_mgr.available_size}")

# 释放一个槽位
table_mgr.free(slots[2])
print(f"释放槽位 {slots[2]} 后可用: {table_mgr.available_size}")

## 6.4 PrefillManager: Prefill 批处理

负责调度 prefill 阶段的请求。

In [ ]:
class Req:
    """简化版 Req"""
    def __init__(self, input_ids, table_idx, cached_len, output_len, uid, sampling_params):
        self.host_ids = input_ids
        self.table_idx = table_idx
        self.cached_len = cached_len
        self.device_len = len(input_ids)
        self.max_device_len = len(input_ids) + output_len
        self.uid = uid
        self.sampling_params = sampling_params
    
    @property
    def extend_len(self) -> int:
        return self.device_len - self.cached_len

class ChunkedReq(Req):
    """分块请求 - 还未完成 prefill"""
    pass

class Batch:
    def __init__(self, reqs, phase):
        self.reqs = reqs
        self.phase = phase

@dataclass
class PrefillManager:
    """Prefill 批处理管理器"""
    pending_list: List[PendingReq] = field(default_factory=list)
    
    def add_one_req(self, uid: int, input_ids: torch.Tensor, sampling_params: SamplingParams):
        """添加新请求到等待队列"""
        self.pending_list.append(PendingReq(uid, input_ids, sampling_params))
    
    def schedule_next_batch(self, token_budget: int) -> Optional[Batch]:
        """调度下一个 prefill 批次"""
        if not self.pending_list:
            return None
        
        reqs = []
        remaining_budget = token_budget
        scheduled = []
        
        for pending in self.pending_list:
            # 计算需要处理的 token 数
            extend_len = pending.input_len
            
            if extend_len <= remaining_budget:
                # 可以完整处理
                remaining_budget -= extend_len
                reqs.append(self._create_req(pending, extend_len))
                scheduled.append(pending)
            elif remaining_budget > 0:
                # 分块处理
                chunk_size = remaining_budget
                reqs.append(self._create_chunked_req(pending, chunk_size))
                remaining_budget = 0
                break
            else:
                break
        
        # 移除已调度的请求
        for s in scheduled:
            self.pending_list.remove(s)
        
        if not reqs:
            return None
        
        return Batch(reqs=reqs, phase="prefill")
    
    def _create_req(self, pending: PendingReq, extend_len: int) -> Req:
        return Req(
            input_ids=pending.input_ids,
            table_idx=0,  # 实际由 TableManager 分配
            cached_len=0,
            output_len=pending.output_len,
            uid=pending.uid,
            sampling_params=pending.sampling_params,
        )
    
    def _create_chunked_req(self, pending: PendingReq, chunk_size: int) -> ChunkedReq:
        return ChunkedReq(
            input_ids=pending.input_ids[:chunk_size],
            table_idx=0,
            cached_len=0,
            output_len=pending.output_len,
            uid=pending.uid,
            sampling_params=pending.sampling_params,
        )
    
    @property
    def runnable(self) -> bool:
        return len(self.pending_list) > 0

In [ ]:
# 演示 Prefill 调度
prefill_mgr = PrefillManager()

# 添加几个请求
prefill_mgr.add_one_req(1, torch.arange(100), SamplingParams(max_tokens=50))
prefill_mgr.add_one_req(2, torch.arange(200), SamplingParams(max_tokens=50))
prefill_mgr.add_one_req(3, torch.arange(150), SamplingParams(max_tokens=50))

print("等待队列:")
for p in prefill_mgr.pending_list:
    print(f"  Req {p.uid}: {p.input_len} tokens")

# 调度第一个批次 (预算 250 tokens)
print("\n调度批次 (预算=250 tokens):")
batch1 = prefill_mgr.schedule_next_batch(token_budget=250)
if batch1:
    for req in batch1.reqs:
        req_type = "ChunkedReq" if isinstance(req, ChunkedReq) else "Req"
        print(f"  {req_type} {req.uid}: {req.extend_len} tokens")

print(f"\n剩余等待请求: {len(prefill_mgr.pending_list)}")

## 6.5 DecodeManager: Decode 批处理

In [ ]:
@dataclass
class DecodeManager:
    """Decode 批处理管理器"""
    running_reqs: List[Req] = field(default_factory=list)
    
    def add_reqs(self, reqs: List[Req]) -> None:
        """添加完成 prefill 的请求"""
        for req in reqs:
            if not isinstance(req, ChunkedReq) and req not in self.running_reqs:
                self.running_reqs.append(req)
    
    def remove_req(self, req: Req) -> None:
        """移除完成的请求"""
        if req in self.running_reqs:
            self.running_reqs.remove(req)
    
    def schedule_next_batch(self) -> Optional[Batch]:
        """调度下一个 decode 批次"""
        # Decode 时每个请求只处理一个 token
        runnable = [r for r in self.running_reqs if r.device_len < r.max_device_len]
        if not runnable:
            return None
        return Batch(reqs=runnable, phase="decode")
    
    @property
    def runnable(self) -> bool:
        return len(self.running_reqs) > 0
    
    @property
    def inflight_tokens(self) -> int:
        """正在进行的请求预计需要的 token 数"""
        return sum(r.max_device_len - r.device_len for r in self.running_reqs)

# 演示
decode_mgr = DecodeManager()

# 模拟完成 prefill 的请求
req1 = Req(torch.arange(10), 0, 0, 20, 1, SamplingParams())
req2 = Req(torch.arange(15), 1, 0, 20, 2, SamplingParams())

decode_mgr.add_reqs([req1, req2])

print(f"Decode 队列: {len(decode_mgr.running_reqs)} 个请求")
print(f"预计剩余 tokens: {decode_mgr.inflight_tokens}")

# 调度 decode 批次
batch = decode_mgr.schedule_next_batch()
if batch:
    print(f"\nDecode 批次:")
    for req in batch.reqs:
        print(f"  Req {req.uid}: 位置 {req.device_len}")

## 6.6 Continuous Batching

传统批处理 vs Continuous Batching:

```
传统批处理:
┌────────────────────────────────────────────────────────────┐
│  Batch 1: Req A, Req B, Req C                              │
│  ├─ A 完成 (10 tokens) ───────────────────── 等待 ────────│
│  ├─ B 完成 (20 tokens) ───────────── 等待 ────────────────│
│  └─ C 完成 (50 tokens) ───────────────────────────────────│
│                                                            │
│  Batch 2: Req D, Req E    ← 必须等 Batch 1 全部完成        │
└────────────────────────────────────────────────────────────┘

Continuous Batching:
┌────────────────────────────────────────────────────────────┐
│  时间 →                                                    │
│  A: ████████                                               │
│  B: ████████████████                                       │
│  C: ██████████████████████████████████████████████         │
│  D:         ████████████████████         ← A完成后立即加入 │
│  E:                     ████████████████ ← B完成后立即加入 │
│                                                            │
│  GPU 利用率更高，延迟更低                                  │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
def simulate_continuous_batching():
    """模拟 Continuous Batching"""
    prefill_mgr = PrefillManager()
    decode_mgr = DecodeManager()
    finished_reqs: Set[int] = set()
    
    # 初始请求
    requests = [
        (1, 50, 10),   # (uid, input_len, output_len)
        (2, 100, 20),
        (3, 30, 15),
    ]
    
    for uid, input_len, output_len in requests:
        prefill_mgr.add_one_req(uid, torch.arange(input_len), SamplingParams(max_tokens=output_len))
    
    print("=== Continuous Batching 模拟 ===")
    step = 0
    
    while prefill_mgr.runnable or decode_mgr.runnable:
        step += 1
        print(f"\n--- Step {step} ---")
        
        # 优先调度 prefill
        if prefill_mgr.runnable:
            batch = prefill_mgr.schedule_next_batch(token_budget=100)
            if batch:
                print(f"Prefill batch: {[r.uid for r in batch.reqs]}")
                # 完成 prefill 后加入 decode 队列
                decode_mgr.add_reqs(batch.reqs)
        
        # 调度 decode
        if decode_mgr.runnable:
            batch = decode_mgr.schedule_next_batch()
            if batch:
                print(f"Decode batch: {[r.uid for r in batch.reqs]}")
                
                # 模拟生成一个 token
                for req in batch.reqs:
                    req.device_len += 1
                    
                    # 检查是否完成
                    if req.device_len >= req.max_device_len:
                        finished_reqs.add(req.uid)
                        decode_mgr.remove_req(req)
                        print(f"  Req {req.uid} 完成!")
        
        if step > 50:  # 防止无限循环
            break
    
    print(f"\n总步数: {step}")
    print(f"完成的请求: {finished_reqs}")

simulate_continuous_batching()

## 6.7 Chunked Prefill

将长输入分块处理，减少峰值内存使用。

```
原始请求: 1000 tokens
max_prefill_length: 256

Chunk 1: tokens[0:256]   → 计算，缓存 KV
Chunk 2: tokens[256:512] → 计算，缓存 KV (复用 Chunk 1 的 KV)
Chunk 3: tokens[512:768] → 计算，缓存 KV (复用 Chunk 1,2 的 KV)
Chunk 4: tokens[768:1000] → 计算，完成 prefill
```

In [ ]:
def simulate_chunked_prefill(input_len: int, max_chunk_size: int):
    """模拟 Chunked Prefill"""
    print(f"输入长度: {input_len} tokens")
    print(f"最大 chunk 大小: {max_chunk_size} tokens")
    print("="*50)
    
    processed = 0
    chunk_id = 1
    
    while processed < input_len:
        chunk_size = min(max_chunk_size, input_len - processed)
        end = processed + chunk_size
        
        print(f"\nChunk {chunk_id}:")
        print(f"  处理范围: [{processed}:{end}]")
        print(f"  chunk 大小: {chunk_size} tokens")
        print(f"  复用已缓存的 KV: {processed} tokens")
        
        processed = end
        chunk_id += 1
        
        if processed >= input_len:
            print(f"\n  ✓ Prefill 完成！开始 decode...")
        else:
            print(f"  → 继续下一个 chunk (ChunkedReq)")

simulate_chunked_prefill(input_len=1000, max_chunk_size=256)

## 6.8 调度策略: Prefill 优先 vs Decode 优先

| 策略 | 特点 | 适用场景 |
|------|------|----------|
| Prefill 优先 | 新请求尽快开始 | 低延迟要求 |
| Decode 优先 | 已有请求尽快完成 | 高吞吐要求 |
| 混合策略 | 平衡延迟和吞吐 | 生产环境 |

Mini-SGLang 默认使用 **Prefill 优先** 策略。

## 6.9 小结

### 核心要点:

1. **Scheduler 的职责**:
   - 接收和管理请求
   - 分配资源 (槽位、KV Cache)
   - 调度批次 (Prefill/Decode)
   - 执行推理和返回结果

2. **资源管理器**:
   - TableManager: 管理 page table 槽位
   - CacheManager: 管理 KV Cache

3. **批处理管理器**:
   - PrefillManager: 调度 prefill 批次
   - DecodeManager: 调度 decode 批次

4. **Continuous Batching**:
   - 请求完成后立即释放资源
   - 新请求可以立即加入
   - 提高 GPU 利用率

5. **Chunked Prefill**:
   - 将长输入分块处理
   - 减少峰值内存使用

---

**下一步**: [Module 7: 完整推理流程](./07_inference_flow.ipynb) - 串联所有组件，理解完整的推理流程。